# Random Forest Iterations

This notebook compares two random forest iterations on the same train/test split used in the grid-search notebook.

In [6]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'pyproject.toml').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config.config import FEATURES, RAW_CSV, TARGET

df = pd.read_csv(RAW_CSV)
X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

labels = sorted(y_test.unique())

In [7]:
def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_test, y_pred),
        'precision_macro': precision_score(y_test, y_pred, average='macro', zero_division=0),
        'recall_macro': recall_score(y_test, y_pred, average='macro', zero_division=0),
        'f1_macro': f1_score(y_test, y_pred, average='macro', zero_division=0),
        'precision_weighted': precision_score(y_test, y_pred, average='weighted', zero_division=0),
        'recall_weighted': recall_score(y_test, y_pred, average='weighted', zero_division=0),
        'f1_weighted': f1_score(y_test, y_pred, average='weighted', zero_division=0),
        'precision_micro': precision_score(y_test, y_pred, average='micro', zero_division=0),
        'recall_micro': recall_score(y_test, y_pred, average='micro', zero_division=0),
        'f1_micro': f1_score(y_test, y_pred, average='micro', zero_division=0),
    }

    print(f'{name} classification report')
    print(classification_report(y_test, y_pred, zero_division=0))

    metrics_df = pd.DataFrame([metrics], index=[name]).T
    confusion_df = pd.DataFrame(
        confusion_matrix(y_test, y_pred, labels=labels),
        index=labels,
        columns=labels,
    )

    display(metrics_df)
    display(confusion_df)

    return model, metrics_df, y_pred

## Iteration 1: Default Random Forest

In [8]:
default_rf = RandomForestClassifier()
default_rf, default_metrics, default_pred = evaluate_model(
    'iteration_1_default',
    default_rf,
    X_train,
    X_test,
    y_train,
    y_test,
)

iteration_1_default classification report
                               precision    recall  f1-score   support

            Basilar-type aura       0.83      0.83      0.83         6
 Familial hemiplegic migraine       0.75      1.00      0.86         3
        Migraine without aura       0.87      1.00      0.93        13
                        Other       1.00      0.50      0.67         4
 Sporadic hemiplegic migraine       0.00      0.00      0.00         2
   Typical aura with migraine       0.98      0.98      0.98        49
Typical aura without migraine       1.00      1.00      1.00         3

                     accuracy                           0.93        80
                    macro avg       0.78      0.76      0.75        80
                 weighted avg       0.92      0.93      0.92        80



,iteration_1_default
accuracy,0.925000
balanced_accuracy,0.758989
precision_macro,0.775656
recall_macro,0.758989
f1_macro,0.752187
precision_weighted,0.918958
recall_weighted,0.925000
f1_weighted,0.916369
precision_micro,0.925000
recall_micro,0.925000


,Basilar-type aura,Familial hemiplegic migraine,Migraine without aura,Other,Sporadic hemiplegic migraine,Typical aura with migraine,Typical aura without migraine
Basilar-type aura,5,1,0,0,0,0,0
Familial hemiplegic migraine,0,3,0,0,0,0,0
Migraine without aura,0,0,13,0,0,0,0
Other,0,0,2,2,0,0,0
Sporadic hemiplegic migraine,1,0,0,0,0,1,0
Typical aura with migraine,0,0,0,0,1,48,0
Typical aura without migraine,0,0,0,0,0,0,3


## Iteration 2: Tuned Random Forest

Best params from `random_forest_grid_search.ipynb`: `max_depth=None`, `max_features='sqrt'`, `min_samples_leaf=1`, `min_samples_split=2`, `n_estimators=100`.

In [9]:
best_rf = RandomForestClassifier(
    max_depth=None,
    max_features='sqrt',
    min_samples_leaf=1,
    min_samples_split=2,
    n_estimators=100,
)

best_rf, best_metrics, best_pred = evaluate_model(
    'iteration_2_best_params',
    best_rf,
    X_train,
    X_test,
    y_train,
    y_test,
)

iteration_2_best_params classification report
                               precision    recall  f1-score   support

            Basilar-type aura       1.00      0.83      0.91         6
 Familial hemiplegic migraine       0.75      1.00      0.86         3
        Migraine without aura       0.87      1.00      0.93        13
                        Other       1.00      0.50      0.67         4
 Sporadic hemiplegic migraine       0.00      0.00      0.00         2
   Typical aura with migraine       0.96      0.98      0.97        49
Typical aura without migraine       1.00      1.00      1.00         3

                     accuracy                           0.93        80
                    macro avg       0.80      0.76      0.76        80
                 weighted avg       0.92      0.93      0.92        80



,iteration_2_best_params
accuracy,0.925000
balanced_accuracy,0.758989
precision_macro,0.796667
recall_macro,0.758989
f1_macro,0.761596
precision_weighted,0.919458
recall_weighted,0.925000
f1_weighted,0.915990
precision_micro,0.925000
recall_micro,0.925000


,Basilar-type aura,Familial hemiplegic migraine,Migraine without aura,Other,Sporadic hemiplegic migraine,Typical aura with migraine,Typical aura without migraine
Basilar-type aura,5,1,0,0,0,0,0
Familial hemiplegic migraine,0,3,0,0,0,0,0
Migraine without aura,0,0,13,0,0,0,0
Other,0,0,2,2,0,0,0
Sporadic hemiplegic migraine,0,0,0,0,0,2,0
Typical aura with migraine,0,0,0,0,1,48,0
Typical aura without migraine,0,0,0,0,0,0,3


## Iteration 3: Remove Hemiplegic Migraine

This edge-case iteration removes every row where the target contains `hemiplegic migraine` before training and evaluation.

In [10]:
no_hemiplegic_df = df[~df[TARGET].str.contains('hemiplegic migraine', case=False)].copy()

X_no_hemiplegic = no_hemiplegic_df[FEATURES]
y_no_hemiplegic = no_hemiplegic_df[TARGET]

X_train_no_hemiplegic, X_test_no_hemiplegic, y_train_no_hemiplegic, y_test_no_hemiplegic = train_test_split(
    X_no_hemiplegic,
    y_no_hemiplegic,
    test_size=0.2,
    random_state=42,
)

no_hemiplegic_rf = RandomForestClassifier(
    max_depth=None,
    max_features='sqrt',
    min_samples_leaf=1,
    min_samples_split=2,
    n_estimators=100,
)

no_hemiplegic_rf, no_hemiplegic_metrics, no_hemiplegic_pred = evaluate_model(
    'iteration_3_no_hemiplegic',
    no_hemiplegic_rf,
    X_train_no_hemiplegic,
    X_test_no_hemiplegic,
    y_train_no_hemiplegic,
    y_test_no_hemiplegic,
)

iteration_3_no_hemiplegic classification report
                               precision    recall  f1-score   support

            Basilar-type aura       1.00      1.00      1.00         2
        Migraine without aura       1.00      1.00      1.00        15
                        Other       1.00      0.67      0.80         6
   Typical aura with migraine       0.96      1.00      0.98        48
Typical aura without migraine       1.00      1.00      1.00         2

                     accuracy                           0.97        73
                    macro avg       0.99      0.93      0.96        73
                 weighted avg       0.97      0.97      0.97        73



,iteration_3_no_hemiplegic
accuracy,0.972603
balanced_accuracy,0.933333
precision_macro,0.992000
recall_macro,0.933333
f1_macro,0.955918
precision_weighted,0.973699
recall_weighted,0.972603
f1_weighted,0.970143
precision_micro,0.972603
recall_micro,0.972603


,Basilar-type aura,Familial hemiplegic migraine,Migraine without aura,Other,Sporadic hemiplegic migraine,Typical aura with migraine,Typical aura without migraine
Basilar-type aura,2,0,0,0,0,0,0
Familial hemiplegic migraine,0,0,0,0,0,0,0
Migraine without aura,0,0,15,0,0,0,0
Other,0,0,0,4,0,2,0
Sporadic hemiplegic migraine,0,0,0,0,0,0,0
Typical aura with migraine,0,0,0,0,0,48,0
Typical aura without migraine,0,0,0,0,0,0,2


## Iteration 4: Remove Defect Cases

This edge-case iteration removes every row where the `Defect` feature is `1` before training and evaluation.

In [11]:
no_defect_df = df[df['Defect'] != 1].copy()

X_no_defect = no_defect_df[FEATURES]
y_no_defect = no_defect_df[TARGET]

X_train_no_defect, X_test_no_defect, y_train_no_defect, y_test_no_defect = train_test_split(
    X_no_defect,
    y_no_defect,
    test_size=0.2,
    random_state=42,
)

no_defect_rf = RandomForestClassifier(
    max_depth=None,
    max_features='sqrt',
    min_samples_leaf=1,
    min_samples_split=2,
    n_estimators=100,
)

no_defect_rf, no_defect_metrics, no_defect_pred = evaluate_model(
    'iteration_4_no_defect',
    no_defect_rf,
    X_train_no_defect,
    X_test_no_defect,
    y_train_no_defect,
    y_test_no_defect,
)

iteration_4_no_defect classification report
                               precision    recall  f1-score   support

            Basilar-type aura       1.00      0.33      0.50         3
 Familial hemiplegic migraine       0.67      0.40      0.50         5
        Migraine without aura       0.89      1.00      0.94        16
                        Other       1.00      0.67      0.80         3
 Sporadic hemiplegic migraine       0.00      0.00      0.00         2
   Typical aura with migraine       0.90      1.00      0.95        46
Typical aura without migraine       1.00      1.00      1.00         4

                     accuracy                           0.90        79
                    macro avg       0.78      0.63      0.67        79
                 weighted avg       0.87      0.90      0.87        79



,iteration_4_no_defect
accuracy,0.898734
balanced_accuracy,0.628571
precision_macro,0.779645
recall_macro,0.628571
f1_macro,0.669947
precision_weighted,0.873997
recall_weighted,0.898734
f1_weighted,0.874528
precision_micro,0.898734
recall_micro,0.898734


,Basilar-type aura,Familial hemiplegic migraine,Migraine without aura,Other,Sporadic hemiplegic migraine,Typical aura with migraine,Typical aura without migraine
Basilar-type aura,1,1,0,0,0,1,0
Familial hemiplegic migraine,0,2,0,0,0,3,0
Migraine without aura,0,0,16,0,0,0,0
Other,0,0,1,2,0,0,0
Sporadic hemiplegic migraine,0,0,1,0,0,1,0
Typical aura with migraine,0,0,0,0,0,46,0
Typical aura without migraine,0,0,0,0,0,0,4


## Comparison

In [12]:
comparison = default_metrics.join([best_metrics, no_hemiplegic_metrics, no_defect_metrics])
comparison['best_params_delta'] = comparison['iteration_2_best_params'] - comparison['iteration_1_default']
comparison['no_hemiplegic_delta'] = comparison['iteration_3_no_hemiplegic'] - comparison['iteration_1_default']
comparison['no_defect_delta'] = comparison['iteration_4_no_defect'] - comparison['iteration_1_default']
comparison

,iteration_1_default,iteration_2_best_params,iteration_3_no_hemiplegic,iteration_4_no_defect,best_params_delta,no_hemiplegic_delta,no_defect_delta
accuracy,0.925000,0.925000,0.972603,0.898734,0.000000,0.047603,-0.026266
balanced_accuracy,0.758989,0.758989,0.933333,0.628571,0.000000,0.174344,-0.130418
precision_macro,0.775656,0.796667,0.992000,0.779645,0.021011,0.216344,0.003989
recall_macro,0.758989,0.758989,0.933333,0.628571,0.000000,0.174344,-0.130418
f1_macro,0.752187,0.761596,0.955918,0.669947,0.009409,0.203732,-0.082239
precision_weighted,0.918958,0.919458,0.973699,0.873997,0.000500,0.054740,-0.044961
recall_weighted,0.925000,0.925000,0.972603,0.898734,0.000000,0.047603,-0.026266
f1_weighted,0.916369,0.915990,0.970143,0.874528,-0.000379,0.053774,-0.041841
precision_micro,0.925000,0.925000,0.972603,0.898734,0.000000,0.047603,-0.026266
recall_micro,0.925000,0.925000,0.972603,0.898734,0.000000,0.047603,-0.026266
